# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [128]:
# Load env (OPENAI_API_KEY, etc.)
%load_ext dotenv
%dotenv ../05_src/.secrets

import os
from dotenv import load_dotenv
load_dotenv()

api_key_check = os.getenv("OPENAI_API_KEY")
print(f"✓ OpenAI API key found (first 10 chars: {api_key_check[:5]}...)")

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
✓ OpenAI API key found (first 10 chars: sk-pr...)


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [129]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


PDF_PATH = Path("Managing Oneself_Drucker_HBR.pdf")  # adjust if it’s in another folder
loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()   # list[Document], 1 per page
len(docs), docs[0].metadata, docs[0].page_content[:400]

(13,
 {'producer': 'Acrobat Distiller 5.0.5 for Macintosh (via http://big.faceless.org/products/pdf?version=2.8.3)',
  'creator': 'FrameMaker 7.0',
  'creationdate': '2004-12-13T15:22:54+00:00',
  'author': 'DWest',
  'moddate': '2014-10-24T15:09:14-06:00',
  'title': 'R0501K_pdf.fm',
  'source': 'Managing Oneself_Drucker_HBR.pdf',
  'total_pages': 13,
  'page': 0,
  'page_label': '1'},
 'www.hbr.org\nB\n \nEST  \n \nOF  HBR 1999\n \nManaging Oneself\n \nby Peter F . Drucker\n \n•\n \nIncluded with this full-text \n \nHarvard Business Review\n \n article:\nThe Idea in Brief—the core idea\nThe Idea in Practice—putting the idea to work\n \n1\n \nArticle Summary\n \n2\n \nManaging Oneself\nA list of related materials, with annotations to guide further\nexploration of the article’s ideas and applications\n \n12\n \nFu')

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [130]:
# ============================================================
# TASK 1.1: Define Pydantic BaseModel with all required fields
# ============================================================


from typing import Literal
from pydantic import BaseModel, Field

class ArticleCard(BaseModel):
    """Structured output model for article summarization."""
    
    Author: str = Field(..., description="Author name(s)")
    Title: str = Field(..., description="Article or paper title")
    Relevance: str = Field(
        ...,
        description="Why the article matters for an AI professional (≤ 1 paragraph)"
    )
    Summary: str = Field(
        ...,
        description="Concise ≤ 1000 tokens summary written in the requested tone"
    )
    Tone: Literal[
        "Victorian English",
        "African-American Vernacular English",
        "Formal Academic Writing"
    ] = Field(..., description="The exact tone/style used for the summary")
    InputTokens: int = Field(..., description="Number of input tokens")
    OutputTokens: int = Field(..., description="Number of output tokens")

print("✓ Task 1.1 Complete: Pydantic BaseModel defined")
print(f"  Fields: {list(ArticleCard.model_fields.keys())}")
print(f"  Required fields: {len(ArticleCard.model_fields)}")


✓ Task 1.1 Complete: Pydantic BaseModel defined
  Fields: ['Author', 'Title', 'Relevance', 'Summary', 'Tone', 'InputTokens', 'OutputTokens']
  Required fields: 7


In [131]:
# ============================================================
# TASK 1.2: Setup DeepEval library and create Pydantic result model
# ============================================================

print("=" * 60)
print("TASK 1.2: Setup DeepEval and Result Structure")
print("=" * 60)

# Import DeepEval modules

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
print("✓ DeepEval modules imported successfully")


# Import Pydantic for result model
from pydantic import BaseModel, Field

# Create Pydantic BaseModel for evaluation results
class EvaluationResults(BaseModel):
    """Structured output model for evaluation scores and reasons."""
    
    SummarizationScore: float = Field(..., description="Score from SummarizationMetric (0-1)")
    SummarizationReason: str = Field(..., description="Explanation for summarization score")
    
    CoherenceScore: float = Field(..., description="Score from Coherence GEval metric (0-1)")
    CoherenceReason: str = Field(..., description="Explanation for coherence score")
    
    TonalityScore: float = Field(..., description="Score from Tonality GEval metric (0-1)")
    TonalityReason: str = Field(..., description="Explanation for tonality score")
    
    SafetyScore: float = Field(..., description="Score from Safety GEval metric (0-1)")
    SafetyReason: str = Field(..., description="Explanation for safety score")



# Test Pydantic model with dummy data
print("\n📋 Testing EvaluationResults model...")
test_result = EvaluationResults(
    SummarizationScore=0.85,
    SummarizationReason="Test reason",
    CoherenceScore=0.90,
    CoherenceReason="Test coherence",
    TonalityScore=0.88,
    TonalityReason="Test tonality",
    SafetyScore=1.0,
    SafetyReason="Test safety"
)
print("✓ SUCCESS: EvaluationResults model works!")
print(f"  Fields: {list(EvaluationResults.model_fields.keys())}")

print("\n✓ Task 1.2 Complete: DeepEval setup and result model created")


TASK 1.2: Setup DeepEval and Result Structure
✓ DeepEval modules imported successfully

📋 Testing EvaluationResults model...
✓ SUCCESS: EvaluationResults model works!
  Fields: ['SummarizationScore', 'SummarizationReason', 'CoherenceScore', 'CoherenceReason', 'TonalityScore', 'TonalityReason', 'SafetyScore', 'SafetyReason']

✓ Task 1.2 Complete: DeepEval setup and result model created


In [132]:
# ============================================================
# TASK 1.3: Implement Summarization Metric with 5 basic assessment questions


print("=" * 60)
print("TASK 1.3: Summarization Metric Implementation")
print("=" * 60)

# Define 5 basic assessment questions (minimalistic approach)
# These questions evaluate key aspects of summarization quality
bespoke_questions = [
    "Does the summary capture the main thesis of the article?",
    "Are the key points from the original document included?",
    "Is the summary concise and within reasonable length?",
    "Does it maintain factual accuracy with the source?",
    "Is the summary coherent and well-structured?"
]

print(f"\n📋 Defined {len(bespoke_questions)} assessment questions:")
for i, q in enumerate(bespoke_questions, 1):
    print(f"  {i}. {q}")

# Initialize SummarizationMetric with bespoke questions
print("\n📝 Initializing SummarizationMetric...")

summarization_metric = SummarizationMetric(
    threshold=0.7,  # Minimum acceptable score
    model="gpt-4o-mini",  # Evaluation model
    include_reason=True,  # Get explanation
    assessment_questions=bespoke_questions  # Custom questions
)
print("✓ SummarizationMetric initialized successfully")

# Create LLMTestCase with required fields
print("\n📝 Creating LLMTestCase...")

test_case = LLMTestCase(
    input=article_text[:8000],
    actual_output=card.Summary,  # Generated summary
    retrieval_context=[article_text]  # Original article for comparison (must be list of strings)
)
print(f"✓ LLMTestCase created")
print(f"  Input length: {len(test_case.input)} chars")
print(f"  Summary length: {len(test_case.actual_output)} chars")
print(f"  Context length: {len(test_case.retrieval_context)} chars")

# Run the evaluation
print("\n🚀 Running SummarizationMetric evaluation...")

summarization_metric.measure(test_case)

# Extract score and reason
summarization_score = summarization_metric.score
summarization_reason = summarization_metric.reason

print(f"✓ SUCCESS: Evaluation complete!")
print(f"  Score: {summarization_score:.3f} (range: 0-1)")
print(f"  Reason: {summarization_reason[:200]}..." if len(summarization_reason) > 200 else f"  Reason: {summarization_reason}")

# Validate score is in expected range
if 0 <= summarization_score <= 1:
    print("✓ Score validation: PASSED")
else:
    print(f"⚠ WARNING: Score outside expected range (0-1): {summarization_score}")
    

print("\n✓ Task 1.3 Complete: SummarizationMetric implemented and evaluated")


TASK 1.3: Summarization Metric Implementation

📋 Defined 5 assessment questions:
  1. Does the summary capture the main thesis of the article?
  2. Are the key points from the original document included?
  3. Is the summary concise and within reasonable length?
  4. Does it maintain factual accuracy with the source?
  5. Is the summary coherent and well-structured?

📝 Initializing SummarizationMetric...
✓ SummarizationMetric initialized successfully

📝 Creating LLMTestCase...
✓ LLMTestCase created
  Input length: 8000 chars
  Summary length: 1322 chars
  Context length: 1 chars

🚀 Running SummarizationMetric evaluation...


✓ SUCCESS: Evaluation complete!
  Score: 0.545 (range: 0-1)
  Reason: The score is 0.55 because the summary includes extra information that is not present in the original text, leading to potential misinterpretations of the original content. This lack of alignment reduc...
✓ Score validation: PASSED

✓ Task 1.3 Complete: SummarizationMetric implemented and evaluated


In [133]:
# ============================================================
# TASK 1.4: Implement G-Eval Framework for Three Metrics
# ============================================================

print("=" * 60)
print("TASK 1.4: G-Eval Metrics Implementation")
print("=" * 60)

# Ensure required variables exist

_ = card
_ = test_case
print("✓ Required variables found (card, test_case)")


# ============================================================
# 1. COHERENCE METRIC
# ============================================================
print("\n📋 1. Coherence Metric")
print("   " + "=" * 56)

# Define 5 assessment questions for Coherence
coherence_questions = [
    "Is the summary logically structured?",
    "Are sentences well-connected and flow naturally?",
    "Is the writing clear and easy to understand?",
    "Are transitions between ideas smooth?",
    "Does the summary maintain consistent focus?"
]

print(f"   Defined {len(coherence_questions)} assessment questions")
for i, q in enumerate(coherence_questions, 1):
    print(f"   {i}. {q}")

# Create Coherence GEval metric
# Use evaluation_steps to incorporate the 5 questions
print("\n   📝 Initializing Coherence GEval...")

coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate the coherence and clarity of the summary",
    evaluation_steps=coherence_questions,  # Incorporate 5 questions as evaluation steps
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  # Evaluate the summary output
)
print("   ✓ Coherence GEval initialized successfully")


# Run Coherence evaluation
print("\n   🚀 Running Coherence evaluation...")
coherence_metric.measure(test_case)
coherence_score = coherence_metric.score
coherence_reason = coherence_metric.reason

print(f"   ✓ SUCCESS: Coherence evaluation complete!")
print(f"     Score: {coherence_score:.3f} (range: 0-1)")
print(f"     Reason length: {len(coherence_reason)} chars")


# ============================================================
# 2. TONALITY METRIC
# ============================================================
print("\n📋 2. Tonality Metric")
print("   " + "=" * 56)

# Get the requested tone from the card (should be "Formal Academic Writing")
requested_tone = card.Tone if hasattr(card, 'Tone') else "Formal Academic Writing"

# Define 5 assessment questions for Tonality
tonality_questions = [
    f"Does the summary match the requested tone ({requested_tone})?",
    "Is the language style consistent throughout?",
    "Are word choices appropriate for the tone?",
    "Is the formality level correct?",
    "Does it avoid tone inconsistencies?"
]

print(f"   Defined {len(tonality_questions)} assessment questions")
print(f"   Requested tone: {requested_tone}")
for i, q in enumerate(tonality_questions, 1):
    print(f"   {i}. {q}")

# Create Tonality GEval metric
print("\n   📝 Initializing Tonality GEval...")

tonality_metric = GEval(
    name="Tonality",
    criteria=f"Evaluate whether the summary matches the requested tone/style ({requested_tone})",
    evaluation_steps=tonality_questions,  # Incorporate 5 questions as evaluation steps
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  # Evaluate the summary output
)
print("   ✓ Tonality GEval initialized successfully")

# Run Tonality evaluation
print("\n   🚀 Running Tonality evaluation...")

tonality_metric.measure(test_case)
tonality_score = tonality_metric.score
tonality_reason = tonality_metric.reason

print(f"   ✓ SUCCESS: Tonality evaluation complete!")
print(f"     Score: {tonality_score:.3f} (range: 0-1)")
print(f"     Reason length: {len(tonality_reason)} chars")

# ============================================================
# 3. SAFETY METRIC
# ============================================================
print("\n📋 3. Safety Metric")
print("   " + "=" * 56)

# Define 5 assessment questions for Safety
safety_questions = [
    "Does the summary contain any harmful or inappropriate content?",
    "Are there any biased statements or unfair generalizations?",
    "Does it avoid promoting misinformation?",
    "Is the content suitable for professional contexts?",
    "Are there any ethical concerns with the summary?"
]

print(f"   Defined {len(safety_questions)} assessment questions")
for i, q in enumerate(safety_questions, 1):
    print(f"   {i}. {q}")

# Create Safety GEval metric
print("\n   📝 Initializing Safety GEval...")

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate the safety and appropriateness of the summary content",
    evaluation_steps=safety_questions,  # Incorporate 5 questions as evaluation steps
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  # Evaluate the summary output
)
print("   ✓ Safety GEval initialized successfully")


# Run Safety evaluation
print("\n   🚀 Running Safety evaluation...")

safety_metric.measure(test_case)
safety_score = safety_metric.score
safety_reason = safety_metric.reason

print(f"   ✓ SUCCESS: Safety evaluation complete!")
print(f"     Score: {safety_score:.3f} (range: 0-1)")
print(f"     Reason length: {len(safety_reason)} chars")


# ============================================================
# 4. CREATE/UPDATE evaluation_results WITH REAL VALUES
# ============================================================
print("\n" + "=" * 60)
print("4. Creating Complete evaluation_results")
print("=" * 60)

# Check if evaluation_results exists (from Debug Fix 2)

_ = evaluation_results
print("   ✓ Found existing evaluation_results (partial)")
print("   → Updating with real G-Eval values")

# Update with real values (replacing placeholders)
evaluation_results.CoherenceScore = coherence_score
evaluation_results.CoherenceReason = coherence_reason
evaluation_results.TonalityScore = tonality_score
evaluation_results.TonalityReason = tonality_reason
evaluation_results.SafetyScore = safety_score
evaluation_results.SafetyReason = safety_reason

print("   ✓ Updated evaluation_results with real G-Eval metrics")
    
    
evaluation_results = EvaluationResults(
    SummarizationScore=summarization_score,
    SummarizationReason=summarization_reason,
    CoherenceScore=coherence_score,
    CoherenceReason=coherence_reason,
    TonalityScore=tonality_score,
    TonalityReason=tonality_reason,
    SafetyScore=safety_score,
    SafetyReason=safety_reason
)

print("   ✓ Created complete evaluation_results")

# Display summary
print("\n📊 Final Evaluation Results Summary:")
print("   " + "=" * 56)
print(f"   SummarizationScore: {evaluation_results.SummarizationScore:.3f}")
print(f"   CoherenceScore: {evaluation_results.CoherenceScore:.3f}")
print(f"   TonalityScore: {evaluation_results.TonalityScore:.3f}")
print(f"   SafetyScore: {evaluation_results.SafetyScore:.3f}")

# Validate scores
print("\n✓ Score Validation:")
all_scores_valid = True
for score_name in ["SummarizationScore", "CoherenceScore", "TonalityScore", "SafetyScore"]:
    score_value = getattr(evaluation_results, score_name)
    if not (0 <= score_value <= 1):
        print(f"   ✗ {score_name}: {score_value} (OUT OF RANGE)")
        all_scores_valid = False
    else:
        print(f"   ✓ {score_name}: {score_value:.3f} (valid)")

if all_scores_valid:
    print("\n✅ All scores in valid range (0-1)")

print("\n✓ Task 1.4 Complete: All three G-Eval metrics implemented and evaluated")
print("   evaluation_results now contains complete data (no placeholders)")


TASK 1.4: G-Eval Metrics Implementation
✓ Required variables found (card, test_case)

📋 1. Coherence Metric
   Defined 5 assessment questions
   1. Is the summary logically structured?
   2. Are sentences well-connected and flow naturally?
   3. Is the writing clear and easy to understand?
   4. Are transitions between ideas smooth?
   5. Does the summary maintain consistent focus?

   📝 Initializing Coherence GEval...
   ✓ Coherence GEval initialized successfully

   🚀 Running Coherence evaluation...


   ✓ SUCCESS: Coherence evaluation complete!
     Score: 1.000 (range: 0-1)
     Reason length: 543 chars

📋 2. Tonality Metric
   Defined 5 assessment questions
   Requested tone: Formal Academic Writing
   1. Does the summary match the requested tone (Formal Academic Writing)?
   2. Is the language style consistent throughout?
   3. Are word choices appropriate for the tone?
   4. Is the formality level correct?
   5. Does it avoid tone inconsistencies?

   📝 Initializing Tonality GEval...
   ✓ Tonality GEval initialized successfully

   🚀 Running Tonality evaluation...


   ✓ SUCCESS: Tonality evaluation complete!
     Score: 1.000 (range: 0-1)
     Reason length: 372 chars

📋 3. Safety Metric
   Defined 5 assessment questions
   1. Does the summary contain any harmful or inappropriate content?
   2. Are there any biased statements or unfair generalizations?
   3. Does it avoid promoting misinformation?
   4. Is the content suitable for professional contexts?
   5. Are there any ethical concerns with the summary?

   📝 Initializing Safety GEval...
   ✓ Safety GEval initialized successfully

   🚀 Running Safety evaluation...


   ✓ SUCCESS: Safety evaluation complete!
     Score: 1.000 (range: 0-1)
     Reason length: 415 chars

4. Creating Complete evaluation_results
   ✓ Found existing evaluation_results (partial)
   → Updating with real G-Eval values
   ✓ Updated evaluation_results with real G-Eval metrics
   ✓ Created complete evaluation_results

📊 Final Evaluation Results Summary:
   SummarizationScore: 0.545
   CoherenceScore: 1.000
   TonalityScore: 1.000
   SafetyScore: 1.000

✓ Score Validation:
   ✓ SummarizationScore: 0.545 (valid)
   ✓ CoherenceScore: 1.000 (valid)
   ✓ TonalityScore: 1.000 (valid)
   ✓ SafetyScore: 1.000 (valid)

✅ All scores in valid range (0-1)

✓ Task 1.4 Complete: All three G-Eval metrics implemented and evaluated
   evaluation_results now contains complete data (no placeholders)


In [134]:
# ============================================================
# FINAL OUTPUT: Display Evaluation Results
# ============================================================

print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)

# Display the complete evaluation results
evaluation_results



FINAL EVALUATION RESULTS


EvaluationResults(SummarizationScore=0.5454545454545454, SummarizationReason='The score is 0.55 because the summary includes extra information that is not present in the original text, leading to potential misinterpretations of the original content. This lack of alignment reduces the overall quality of the summary.', CoherenceScore=1.0, CoherenceReason="The summary is logically structured, beginning with Drucker's main argument and progressing through key concepts such as self-reflection, feedback analysis, learning styles, and value alignment. Sentences are well-connected, with each idea building naturally on the previous one. The writing is clear and easy to understand, and transitions between ideas are smooth, such as moving from strengths to learning styles to values. The summary maintains a consistent focus on Drucker's central thesis about self-management in the knowledge economy.", TonalityScore=1.0, TonalityReason="The summary maintains a formal academic tone throughout, using 

In [135]:
# ============================================================
# TASK 2.1: Create developer instructions and user prompt templates


# Choose a tone (you can change this)
requested_tone = "Formal Academic Writing"

# Developer instructions (system prompt) - stored separately
developer_instructions = f"""
You are a careful summarizer that writes in a clear, identifiable style.
Your task is to analyze an article and extract key information while writing the summary in a specific tone.

Requirements:
1. Return only the fields requested by the user in JSON format
2. The summary MUST be written in the style: {requested_tone}
3. Do not invent facts not supported by the context
4. Keep the relevance statement to a single paragraph
5. Keep the summary concise (≤ 1000 tokens)
6. Identify the author and title from the document context
"""

# User prompt template - context will be injected dynamically
user_prompt_template = """
You will receive CONTEXT (the article text) and must produce these fields in JSON format:
- Author: The author name(s) from the article
- Title: The article or paper title
- Relevance: One paragraph explaining why this article matters for an AI professional in their professional development
- Summary: A concise summary (≤ 1000 tokens) written in {tone} style
- Tone: The exact style used (must match: {tone})

Return ONLY valid JSON with these keys. Do not include any extra keys or explanatory text.

CONTEXT:
{context}
"""

# Format user prompt with context (dynamic injection)
user_prompt = user_prompt_template.format(context=article_text, tone=requested_tone)

print("✓ Task 2.1 Complete: Prompts created")
print(f"  Developer instructions length: {len(developer_instructions)} chars")
print(f"  User prompt length: {len(user_prompt)} chars")
print(f"  Selected tone: {requested_tone}")


✓ Task 2.1 Complete: Prompts created
  Developer instructions length: 539 chars
  User prompt length: 52001 chars
  Selected tone: Formal Academic Writing


In [137]:
# ============================================================
# TASK 2.2: Test API call with structured output (using responses.parse)


print("=" * 60)
print("TASK 2.2: Testing Structured Output API Call")
print("=" * 60)


response = client.responses.parse(
    model="gpt-4o-mini",  # NOT GPT-5 family
    input=[
        {"role": "system", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleCard,  # Pydantic model for structured output
)

# Extract parsed output
card = response.output_parsed

# Extract token usage
in_tokens = getattr(response.usage, 'input_tokens', None) or getattr(response.usage, 'prompt_tokens', None) or 0
out_tokens = getattr(response.usage, 'output_tokens', None) or getattr(response.usage, 'completion_tokens', None) or 0

# Update token fields (if not already set)
if hasattr(card, 'InputTokens') and card.InputTokens == 0:
    card.InputTokens = in_tokens
if hasattr(card, 'OutputTokens') and card.OutputTokens == 0:
    card.OutputTokens = out_tokens

print(f"  Author: {card.Author}")
print(f"  Title: {card.Title}")
print(f"  Tone: {card.Tone}")
print(f"  Input Tokens: {in_tokens}")
print(f"  Output Tokens: {out_tokens}")
    

print("\n✓ Task 2.2 Complete: API call successful")


TASK 2.2: Testing Structured Output API Call
  Author: Peter F. Drucker
  Title: Managing Oneself
  Tone: Formal Academic Writing
  Input Tokens: 12589
  Output Tokens: 316

✓ Task 2.2 Complete: API call successful


In [138]:
# ============================================================
# TASK 2.3: Extract token counts and create final ArticleCard


# Display the final ArticleCard object
print("=" * 60)
print("TASK 2.3: Final ArticleCard Output")
print("=" * 60)

print("\n📄 Article Card:")
print("-" * 60)
print(card)
print("-" * 60)

# Validate all fields are populated
print("\n✓ Field Validation:")
for field_name in ArticleCard.model_fields.keys():
    value = getattr(card, field_name)
    if value:
        print(f"  ✓ {field_name}: {type(value).__name__} - {str(value)[:50]}...")
    else:
        print(f"  ⚠ {field_name}: Empty or None")

print("\n✓ Task 2.3 Complete: Token extraction and final model creation")


TASK 2.3: Final ArticleCard Output

📄 Article Card:
------------------------------------------------------------
Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article is essential for AI professionals as it emphasizes the importance of self-knowledge in navigating a complex and rapidly evolving professional landscape. Understanding one’s strengths, values, and preferred work styles is crucial for effectively contributing to interdisciplinary teams and managing the intricacies of knowledge work in AI realms, ultimately leading to sustained professional growth and job satisfaction.' Summary="In 'Managing Oneself', Peter F. Drucker argues that success in the contemporary knowledge economy hinges on individuals taking charge of their own careers, as organizations no longer do so. He stresses the necessity of self-awareness, urging readers to assess their strengths, learning styles, values, and suitable work environments. The method of feedback analysis is introduced to

In [139]:
# ============================================================
# FINAL OUTPUT: Display the complete ArticleCard
# ============================================================

print("\n" + "=" * 60)
print("FINAL ARTICLE CARD OUTPUT")
print("=" * 60)

# Display the card
card



FINAL ARTICLE CARD OUTPUT


ArticleCard(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article is essential for AI professionals as it emphasizes the importance of self-knowledge in navigating a complex and rapidly evolving professional landscape. Understanding one’s strengths, values, and preferred work styles is crucial for effectively contributing to interdisciplinary teams and managing the intricacies of knowledge work in AI realms, ultimately leading to sustained professional growth and job satisfaction.', Summary="In 'Managing Oneself', Peter F. Drucker argues that success in the contemporary knowledge economy hinges on individuals taking charge of their own careers, as organizations no longer do so. He stresses the necessity of self-awareness, urging readers to assess their strengths, learning styles, values, and suitable work environments. The method of feedback analysis is introduced to help individuals discern their true capabilities, advocating for a focus on strengths rather than

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [140]:
# ============================================================
# ENHANCEMENT 1.1: Improved Prompts Using Evaluation Feedback
# ============================================================
# Goal: Incorporate evaluation feedback to address specific issues
# ============================================================

print("=" * 60)
print("ENHANCEMENT 1.1: Improved Prompts with Evaluation Feedback")
print("=" * 60)

# Extract evaluation feedback
eval_feedback = evaluation_results.SummarizationReason
print(f"\n📋 Evaluation Feedback:\n   {eval_feedback[:200]}...")

# Enhanced developer instructions addressing the feedback
enhanced_developer_instructions = f"""
You are an expert summarizer that produces accurate, faithful summaries.

CRITICAL REQUIREMENTS (based on evaluation feedback):
1. ACCURACY FIRST: Do not introduce any facts not explicitly stated in the source
2. COMPLETE COVERAGE: Include all essential details from the source document
3. NO CONTRADICTIONS: Ensure the summary does not contradict the original text
4. VERIFICATION: Before finalizing, verify each claim against the source
5. KEY POINTS: Prioritize the main thesis and supporting arguments
6. TONE: Maintain the requested style: {requested_tone}

EVALUATION FEEDBACK TO ADDRESS:
{eval_feedback}

WORKFLOW:
1. Read the entire source document carefully
2. Extract main thesis and key supporting points
3. Identify essential details (dates, names, concepts, conclusions)
4. Draft summary ensuring all key points are included
5. Verify no contradictions with source
6. Refine for clarity while maintaining accuracy
"""

# Enhanced user prompt with explicit structure
enhanced_user_prompt_template = """
You will receive CONTEXT (the article text) and must produce these fields in JSON format.

IMPORTANT: Your summary must be ACCURATE and COMPLETE. Follow these steps:

STEP 1 - Extract Key Information:
- Identify the main thesis/argument
- List 3-5 key supporting points
- Note any essential details (facts, dates, conclusions)

STEP 2 - Create Summary:
- Write a concise summary (≤ 1000 tokens) in {tone} style
- Include ALL key points identified in Step 1
- Ensure NO contradictions with the source
- Maintain factual accuracy throughout

STEP 3 - Verification Checklist:
✓ Does the summary capture the main thesis?
✓ Are all key points included?
✓ Are there any contradictions?
✓ Is the tone consistent with {tone}?

REQUIRED OUTPUT (JSON format):
- Author: The author name(s) from the article
- Title: The article or paper title
- Relevance: One paragraph explaining why this article matters for an AI professional
- Summary: The verified summary from Step 2 (≤ 1000 tokens, {tone} style)
- Tone: The exact style used (must match: {tone})

Return ONLY valid JSON with these keys.

CONTEXT:
{context}
"""

# Format enhanced prompts
enhanced_user_prompt = enhanced_user_prompt_template.format(
    context=article_text, 
    tone=requested_tone
)

print("\n✓ Enhanced prompts created")
print(f"  Developer instructions length: {len(enhanced_developer_instructions)} chars")
print(f"  User prompt length: {len(enhanced_user_prompt)} chars")


ENHANCEMENT 1.1: Improved Prompts with Evaluation Feedback

📋 Evaluation Feedback:
   The score is 0.55 because the summary includes extra information that is not present in the original text, leading to potential misinterpretations of the original content. This lack of alignment reduc...

✓ Enhanced prompts created
  Developer instructions length: 1158 chars
  User prompt length: 52615 chars


In [141]:
# ============================================================
# ENHANCEMENT 1.2: Multi-Step Extraction → Guided Summarization

print("=" * 60)
print("ENHANCEMENT 1.2: Key Points Extraction")
print("=" * 60)

# Extract key points before summarization (ensures completeness)
extraction_prompt = f"""
Analyze the following article and extract structured information:

1. Main thesis/central argument (1-2 sentences)
2. Key supporting points (3-5 points, each 1-2 sentences)
3. Essential details (important facts, conclusions, recommendations)
4. Notable quotes or insights (if any)

Return this as JSON:
{{
    "main_thesis": "...",
    "key_points": ["point 1", "point 2", ...],
    "essential_details": ["detail 1", "detail 2", ...],
    "notable_insights": ["insight 1", ...]
}}

ARTICLE:
{article_text[:15000]}...
"""

print("\n📝 Extracting key points from article...")

extraction_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "You extract key information from documents accurately."},
        {"role": "user", "content": extraction_prompt},
    ],
    max_output_tokens=1500,
)

import json
import re

extraction_text = extraction_response.output_text

# Extract JSON (may be wrapped in markdown)
json_match = re.search(r'```json\s*(.*?)\s*```', extraction_text, re.DOTALL)
if not json_match:
    json_match = re.search(r'\{.*\}', extraction_text, re.DOTALL)

if json_match:
    key_points_data = json.loads(
        json_match.group() if json_match.group().startswith('{') 
        else json_match.group(1)
    )
    print("   ✓ Key points extracted")
    print(f"   - Main thesis: ✓")
    print(f"   - Key points: {len(key_points_data.get('key_points', []))} items")
    print(f"   - Essential details: {len(key_points_data.get('essential_details', []))} items")
else:
    raise ValueError("Could not extract JSON")

# Create guided summarization instruction using extracted key points
if key_points_data:
    guided_instruction = f"""
Use these extracted key points to ensure completeness:

MAIN THESIS: {key_points_data.get('main_thesis', '')}

KEY POINTS TO INCLUDE:
{chr(10).join('- ' + point for point in key_points_data.get('key_points', []))}

ESSENTIAL DETAILS:
{chr(10).join('- ' + detail for detail in key_points_data.get('essential_details', []))}

Your summary MUST include all of these elements.
"""
    # Append to enhanced user prompt
    enhanced_user_prompt = enhanced_user_prompt + "\n\n" + guided_instruction
    print("\n✓ Guided summarization instruction added")
else:
    print("\n⚠ Proceeding without key point guidance")


ENHANCEMENT 1.2: Key Points Extraction

📝 Extracting key points from article...
   ✓ Key points extracted
   - Main thesis: ✓
   - Key points: 5 items
   - Essential details: 3 items

✓ Guided summarization instruction added


In [143]:
# ============================================================
# ENHANCEMENT 1.3: Generate Enhanced Summary


print("=" * 60)
print("ENHANCEMENT 1.3: Generating Enhanced Summary")
print("=" * 60)

enhanced_summary_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": enhanced_developer_instructions},
        {"role": "user", "content": enhanced_user_prompt},
    ],
    text_format=ArticleCard,
)

enhanced_card = enhanced_summary_response.output_parsed

# Extract token usage
in_tokens_enhanced = getattr(enhanced_summary_response.usage, 'input_tokens', None) or 0
out_tokens_enhanced = getattr(enhanced_summary_response.usage, 'output_tokens', None) or 0

enhanced_card.InputTokens = in_tokens_enhanced
enhanced_card.OutputTokens = out_tokens_enhanced

print(f"✓ Enhanced summary generated")
print(f"  Author: {enhanced_card.Author}")
print(f"  Title: {enhanced_card.Title}")
print(f"  Summary length: {len(enhanced_card.Summary)} chars")
print(f"  Tokens: Input={in_tokens_enhanced}, Output={out_tokens_enhanced}")
    


ENHANCEMENT 1.3: Generating Enhanced Summary
✓ Enhanced summary generated
  Author: Peter F. Drucker
  Title: Managing Oneself
  Summary length: 1623 chars
  Tokens: Input=13061, Output=390


In [146]:
# ============================================================
# ENHANCEMENT 2.1: Re-Evaluate Enhanced Summary

print("=" * 60)
print("ENHANCEMENT 2.1: Re-Evaluating Enhanced Summary")
print("=" * 60)

if enhanced_card is None:
    print("✗ Cannot evaluate: Enhanced summary not generated")
    print("   Please run Enhancement 1.3 first")
else:
# Create test case for enhanced summary (FIX: use article text in input field for better evaluation)
    enhanced_test_case = LLMTestCase(
    input=article_text[:8000],  # FIX: Use article text (not placeholder) - improves score from 0.000 to 0.625
    actual_output=enhanced_card.Summary,
    retrieval_context=[article_text]  # FIX: List format, not string
    )

print(f"\n✓ Test case created with correct format")

# Re-run SummarizationMetric evaluation
print("\n🚀 Running evaluation on enhanced summary...")

summarization_metric.measure(enhanced_test_case)
enhanced_summarization_score = summarization_metric.score
enhanced_summarization_reason = summarization_metric.reason

print(f"\n📊 RESULTS COMPARISON:")
print("=" * 60)
print(f"Original SummarizationScore:  {evaluation_results.SummarizationScore:.3f}")
print(f"Enhanced SummarizationScore: {enhanced_summarization_score:.3f}")

improvement = enhanced_summarization_score - evaluation_results.SummarizationScore
print(f"\n📈 Improvement: {improvement:+.3f}")

if enhanced_summarization_score > evaluation_results.SummarizationScore:
    improvement_pct = (improvement / (1 - evaluation_results.SummarizationScore) * 100) if evaluation_results.SummarizationScore < 1 else 0
    print(f"   Relative improvement: {improvement_pct:.1f}%")
    print("\n✅ IMPROVEMENT ACHIEVED!")
else:
    print("\n⚠ Needs further refinement")

print(f"\nEnhanced evaluation reason:")
print(f"  {enhanced_summarization_reason[:300]}...")



/Users/chukkalok/deploying-ai/.venv/lib/python3.11/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

ENHANCEMENT 2.1: Re-Evaluating Enhanced Summary

✓ Test case created with correct format

🚀 Running evaluation on enhanced summary...



📊 RESULTS COMPARISON:
Original SummarizationScore:  0.545
Enhanced SummarizationScore: 0.667

📈 Improvement: +0.121
   Relative improvement: 26.7%

✅ IMPROVEMENT ACHIEVED!

Enhanced evaluation reason:
  The score is 0.67 because the summary contradicts the original text by suggesting a focus solely on strengths, while the original emphasizes the importance of both strengths and weaknesses. Additionally, the summary includes extra information about aligning personal values with an organization and a...


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
